In [1]:
import torch
import random
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

d:\miniconda\envs\tinkoff_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Константы
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'ai-forever/ruRoberta-large'
MLM_MODEL_NAME = 'cointegrated/rubert-tiny'
RANDOM_STATE = 42
HYPERPARAMS = {
    'lr': 0.00005,
    'weight_decay': 0.01,
    'betas': (0.9, 0.9),
    'num_epochs': 5,
    'batch_size': 8,
    'warmup_ratio': 0.1,
    'eval_steps': 100,
    'max_grad_norm': 1.0
}

### Читаем данные

In [3]:
# Читаем доступные категории
with open('data/categories.txt', 'r') as f:
    categories = f.readlines()
for i in range(len(categories)):
    categories[i] = categories[i].replace('\n', '')
map_categories = {}
for i in range(len(categories)):
    map_categories[categories[i]] = i
print(map_categories)

{'бытовая техника': 0, 'обувь': 1, 'одежда': 2, 'посуда': 3, 'текстиль': 4, 'товары для детей': 5, 'украшения и аксессуары': 6, 'электроника': 7, 'нет товара': 8}


In [4]:
# Читаем размеченные и сгенерированные данные
marked_data = pd.read_csv('data/marked_data.csv')
generated_data = pd.read_csv('data/generated_data.csv')

# Исправляем ошибки автоматической разметки
marked_data = marked_data[~marked_data['category'].isin(['бытовая техника', 'электроника', 'нет категории'])]
marked_data.loc[marked_data['category'] == 'посуда', 'category'] = 'одежда'
marked_data = marked_data.rename(columns={'text': 'review'})

# Добавляем колонку с источником данных
marked_data['source'] = ['original'] * len(marked_data)
generated_data['source'] = ['generated'] * len(generated_data)

# Соединяем данные
data = pd.concat([marked_data, generated_data], axis=0)

In [5]:
# Распределение данных по категориям
print(data['category'].value_counts())

category
одежда                    1096
нет товара                 740
текстиль                   411
обувь                      333
украшения и аксессуары     322
товары для детей           302
посуда                     300
бытовая техника            298
электроника                281
Name: count, dtype: int64


In [6]:
# Распределение данных по источнику
print(data['source'].value_counts())

source
generated    2272
original     1811
Name: count, dtype: int64


### Подготовка данных

In [7]:
# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(categories), dtype='auto', device_map='auto')

# Загружаем модель и токенизатор для перевода на английский
ru_en_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ru-en")
ru_en_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ru-en")

# Загружаем модель и токенизатор для перевода на русский
en_ru_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
en_ru_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-ru")

# Загружаем модель и токенизатор для аугментации <вставка синонима> и для аугментации <контекстная замена>
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_MODEL_NAME)
mlm_model = AutoModelForMaskedLM.from_pretrained(MLM_MODEL_NAME)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\miniconda\envs\tinkoff_env\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [8]:
# Датасет отзывов
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, synonym_aug=None, mask_aug=None, backtranslation_aug=None, aug_prob=0.5):
        self.texts = texts
        self.labels = labels
        self.synonym_aug = synonym_aug
        self.mask_aug = mask_aug
        self.backtranslation_aug = backtranslation_aug
        self.aug_prob = aug_prob

        self.augs = []
        if self.synonym_aug:
            self.augs.append(self.synonym_aug)
        if self.mask_aug:
            self.augs.append(self.mask_aug)
        if self.backtranslation_aug:
            self.augs.append(self.backtranslation_aug)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]
        text = self.texts[idx]

        if self.augs and random.random() < self.aug_prob:
            aug = random.choice(self.augs)
            text = aug(text)
        return text, label

# Аугментация перевода   
def backtranslation(text):
    # Перевод на английский
    inputs = ru_en_tokenizer(text, return_tensors='pt')
    output = ru_en_model.generate(**inputs)
    english_text = ru_en_tokenizer.batch_decode(output, skip_special_tokens=True)

    # Перевод на русский
    inputs = en_ru_tokenizer(english_text, return_tensors='pt')
    output = en_ru_model.generate(**inputs)
    russian_text = en_ru_tokenizer.batch_decode(output, skip_special_tokens=True)

    # Возвращаем результат
    return russian_text[0]

# Аугментация вставки синонима    
def insert_synonym(text):
    words = text.split(' ')
    target_idx = random.randint(0, len(words) - 1)

    # Добавление маски
    new_words = words.copy()
    new_words.insert(target_idx + 1, tokenizer.mask_token)
    masked_text = ' '.join(new_words)  

    # Определяем позиции маски
    encoding = mlm_tokenizer(masked_text, return_tensors='pt')
    input_ids = encoding['input_ids'][0]
    attention_mask = encoding['attention_mask'][0]
    mask_token_index = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

    # Предсказываем новые токены
    with torch.no_grad():
        outputs = mlm_model(input_ids=input_ids.unsqueeze(0), attention_mask=attention_mask.unsqueeze(0))
    logits = outputs.logits[0]    
    masked_input_ids = input_ids.clone()
    for pos in mask_token_index:
        predicted_token_id = logits[pos].argmax().item()
        masked_input_ids[pos] = predicted_token_id

    # Получаем новый текст
    new_text = mlm_tokenizer.decode(masked_input_ids, skip_special_tokens=True)
    return new_text        

# Аугментация контекстной замены
def mask_augmentation(text, mask_ratio=0.15, min_words=4):
    words = text.split(' ')
    if len(words) < min_words:
        return text
    
    # Токенизация отзыва
    tokenized = mlm_tokenizer(text, return_tensors='pt', add_special_tokens=True)
    input_ids = tokenized['input_ids'][0]
    attention_mask = tokenized['attention_mask'][0]
    candidate_positions = [i for i, token in enumerate(input_ids) if token not in [tokenizer.cls_token_id, tokenizer.sep_token_id]]

    # Маскирование токенов
    n_mask = max(1, int(len(candidate_positions) * mask_ratio))
    mask_positions = random.sample(candidate_positions, n_mask)
    masked_input_ids = input_ids.clone()
    for pos in mask_positions:
        masked_input_ids[pos] = tokenizer.mask_token_id

    # Предсказываем новые токены
    with torch.no_grad():
        outputs = mlm_model(input_ids=masked_input_ids.unsqueeze(0), attention_mask=attention_mask.unsqueeze(0))
    logits = outputs.logits[0]
    for pos in mask_positions:
        predicted_token_id = logits[pos].argmax().item()
        masked_input_ids[pos] = predicted_token_id

    # Получаем новый текст
    new_text = mlm_tokenizer.decode(masked_input_ids, skip_special_tokens=True)
    return new_text

In [9]:
# Подготовка батча перед подачей в модель
def collate_fn(batch, tokenizer, map_categories, device):
    reviews = [review[0] for review in batch]
    input_ids = [tokenizer(review, add_special_tokens=True, return_tensors='pt')['input_ids'].reshape(-1) for review in reviews]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
    attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)
    labels = torch.tensor([map_categories[review[1]] for review in batch]).long().to(device)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [10]:
# Разбиваем данные на train и test сохраняя распределение категории и источников 
data['stratify'] = data['category'] + '_' + data['source']
train, test = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE, stratify=data['stratify'])
X_train, y_train = train['review'].to_list(), train['category'].to_list()
X_test, y_test = test['review'].to_list(), test['category'].to_list()

# Создаем датасеты и даталоадеры
train_dataset = ReviewDataset(X_train, y_train, insert_synonym, mask_augmentation, backtranslation)
test_dataset = ReviewDataset(X_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=True,
                             collate_fn=lambda x: collate_fn(x, tokenizer, map_categories, DEVICE))
test_dataloader = DataLoader(test_dataset, batch_size=HYPERPARAMS['batch_size'], shuffle=True,
                            collate_fn=lambda x: collate_fn(x, tokenizer, map_categories, DEVICE))

In [11]:
# Инициализация оптимизатора
optimizer = torch.optim.AdamW(model.parameters(), lr=HYPERPARAMS['lr'], betas=HYPERPARAMS['betas'], weight_decay=HYPERPARAMS['weight_decay'])

# Инициализация скедулера для lr
total_steps = HYPERPARAMS['num_epochs'] * len(train_dataloader)
warmup_steps = int(total_steps * HYPERPARAMS['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

# Загрузка лосс функции
loss_fn = torch.nn.CrossEntropyLoss()

In [12]:
best_f1 = 0.0
for epoch in range(HYPERPARAMS['num_epochs']):

    # Тренировочный цикл
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_dataloader, total=len(train_dataloader), desc=f'Epoch {epoch + 1} training'):
        optimizer.zero_grad()

        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), HYPERPARAMS['max_grad_norm'])
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
    train_loss /= len(train_dataloader)

    # Валидационный цикл
    model.eval()
    all_preds = []
    all_labels = []
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(test_dataloader, total=len(test_dataloader), desc=f'Epoch {epoch + 1} validation'):
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item() 

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    val_loss /= len(test_dataloader)
    f1 = f1_score(all_labels, all_preds, average='weighted')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'output_files/best_model.pt')

    print(f'Train Loss: {train_loss}; Eval Loss: {val_loss}; F1: {f1}')

Epoch 1 training:   0%|          | 2/409 [00:16<56:33,  8.34s/it]


KeyboardInterrupt: 